In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/the-ancient-texts-provenance-challenge/sample_submission.csv
/kaggle/input/the-ancient-texts-provenance-challenge/train.csv
/kaggle/input/the-ancient-texts-provenance-challenge/test.csv


# Installing Dependencies

I've commented them since they sometimes prompt restarting kernal.

In [2]:
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source 

In [3]:
#!pip install  evaluate

In [4]:
#from huggingface_hub import notebook_login

#notebook_login()

# Data Loading

In [5]:
df=pd.read_csv("/kaggle/input/the-ancient-texts-provenance-challenge/train.csv")

In [6]:
from datasets import Dataset

In [7]:
ds=Dataset.from_pandas(df)

In [8]:
model_id="google-bert/bert-base-multilingual-uncased"

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True,padding=True)

In [11]:
tokenized_ds = ds.map(preprocess_function, batched=True,remove_columns=['text','id'])

Map:   0%|          | 0/119656 [00:00<?, ? examples/s]

In [12]:
tokenized_ds=tokenized_ds.train_test_split(test_size=0.3,seed=42)

# Importing Model 

Since I have used a fairly small model, LoRA did not seem neccessary. 

In [13]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer,padding=True)

2025-10-20 05:10:12.042100: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760937012.391831      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760937012.500793      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [14]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=15)

model.safetensors:   0%|          | 0.00/672M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(105879, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [16]:
model.num_parameters()/(10**6)

167.367951

# Compute Metric

Since the competition requires f1-score, I've done this. 

In [17]:
#import evaluate

#f1_metric = evaluate.load("f1")
#def compute_metrics(eval_pred):
#    logits, labels = eval_pred
#    predictions = logits.argmax(axis=-1)
#    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
#    return f1


# Model training

This takes a lot of time, Hence I have trained in this notebook and I pushed my model to hf_hub

In [18]:
training_args = TrainingArguments( output_dir='bert_multi',
                                 num_train_epochs=1,
                                  per_device_train_batch_size=12,
                                  per_device_eval_batch_size=12,
                                  bf16=False,
                                  fp16=True,
                                  tf32=False,
                                  gradient_accumulation_steps=1,
                                  adam_beta1=0.9,
                                  adam_beta2=0.999,
                                  learning_rate=2e-5,
                                  weight_decay=0.01,
                                  logging_dir='logs',
                                  logging_strategy="steps",
                                  logging_steps = 100,
                                  save_steps=1000,
                                  save_total_limit=20,
                                  report_to='none',
                                )

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    data_collator=data_collator,
    #compute_metrics=compute_metrics
)

In [20]:
#result=trainer.train()

In [21]:
#trainer.push_to_hub()

In [22]:
#print(result)

Installing the evaluate model needs to be done carefully and I could not find a work around, so I used it to train the model and commented these parts out.

Pushed the model to the hub. 

# Prediction

In [23]:
from transformers import pipeline

classifier = pipeline("text-classification", model="powervel/bert_multi",truncation=True,padding=True)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


In [24]:
tdf=pd.read_csv("/kaggle/input/the-ancient-texts-provenance-challenge/test.csv")

In [25]:
pred=[]
for i in range(len(tdf)):
    if i%1000==0:
        print(i,"th round")
    pred.append(int(classifier(tdf['text'][i])[0]['label'][6:]))

0 th round


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


1000 th round
2000 th round
3000 th round
4000 th round
5000 th round
6000 th round
7000 th round
8000 th round
9000 th round
10000 th round
11000 th round
12000 th round
13000 th round
14000 th round
15000 th round
16000 th round
17000 th round
18000 th round
19000 th round
20000 th round
21000 th round
22000 th round
23000 th round
24000 th round
25000 th round
26000 th round
27000 th round
28000 th round
29000 th round


In [26]:
len(pred)==len(tdf)

True

# Submission

In [27]:
y_pred=pd.DataFrame(pred,columns=['label'])
sub=pd.concat([tdf['id'],y_pred],axis=1)
sub.to_csv("submission.csv",index=False)